# SVARA — Google Colab GPU Training Runner
**Smart Voice Assistant for Residential Automation**  
Informatics Engineering, Universitas Mercu Buana  

This notebook trains and benchmarks the main model **M0 (`facebook/wav2vec2-base`)** on Google Colab GPU.

### Prerequisites:
- Colab Runtime set to **GPU (T4 or A100)** (`Runtime > Change runtime type > T4 GPU`).
- Fluent Speech Commands dataset placed in Google Drive at `MyDrive/SVARA/data/raw/fluent_speech_commands_dataset` (or downloaded directly).

## 1. Environment & GPU Check

In [ ]:
!nvidia-smi

## 2. Mount Google Drive & Clone/Update SVARA Repository

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

import os
# Set HuggingFace cache directory on Drive to avoid re-downloading checkpoints
os.environ['HF_HOME'] = '/content/drive/MyDrive/SVARA/cache/huggingface'
os.makedirs(os.environ['HF_HOME'], exist_ok=True)

%cd /content
if not os.path.exists('SVARA'):
    !git clone https://github.com/YogUNI/SVARA.git
    %cd SVARA
else:
    %cd SVARA
    !git pull origin main

## 3. Install Requirements

In [ ]:
!pip install -r requirements.txt
!pip install -e .

## 4. Link or Copy Dataset & Splits
Symlink the raw FSC dataset and pre-computed splits from Drive if already extracted.

In [ ]:
import os
# Check if dataset exists in Drive
drive_dataset = '/content/drive/MyDrive/SVARA/data/raw/fluent_speech_commands_dataset'
local_data_dir = 'data/raw/fluent_speech_commands_dataset'

if os.path.exists(drive_dataset) and not os.path.exists(local_data_dir):
    os.makedirs('data/raw', exist_ok=True)
    !ln -s "$drive_dataset" "$local_data_dir"
    print("Linked dataset from Google Drive successfully!")
else:
    print("Local dataset or drive status:", os.path.exists(local_data_dir))

# Run data check audit
!python scripts/00_check_data.py --root data/raw/fluent_speech_commands_dataset
# Generate splits if not already present
!python scripts/01_make_splits.py --config configs/splits.yaml

## 5. Sanity Checks Verification (docs/11 V-10..V-20)
Ensure zero silent bugs before commencing long GPU training.

In [ ]:
!python scripts/sanity_checks.py --config configs/model_w2v2_base.yaml

## 6. Train M0 (`facebook/wav2vec2-base`) — Split A & Split B1 (Tasks P3-07, P3-08)
The training loop is resumable (`--resume`) and automatically saves checkpoints to `reports/runs/`.

In [ ]:
# Train M0 on Split A (Original benchmark)
!python scripts/train.py --config configs/model_w2v2_base.yaml --split A --seed 42

In [ ]:
# Train M0 on Split B1 (Unseen-utterance generalization benchmark)
!python scripts/train.py --config configs/model_w2v2_base.yaml --split B1 --seed 42

## 7. Evaluate Runs (Task P3-05)
Compute exact-match accuracy, per-slot accuracy, bootstrap 95% CI, and confusion matrices.

In [ ]:
import glob
runs = sorted(glob.glob('reports/runs/*_wav2vec2_*'))
for r in runs:
    print(f"Evaluating {r}...")
    split = 'A' if '_A_' in r else 'B1'
    !python scripts/evaluate.py --run "$r" --split "$split" --set test

## 8. Backup Runs & Checkpoints to Google Drive
Ensures training results and checkpoints are safe and persistent.

In [ ]:
!mkdir -p /content/drive/MyDrive/SVARA/reports/
!cp -r reports/runs /content/drive/MyDrive/SVARA/reports/
print("Backed up all run logs and checkpoints to Google Drive successfully!")